Parámetros y rutas

In [1]:
from pathlib import Path

# Proyecto: notebook está en /prueba
ROOT = Path("..")          # sube un nivel a la raíz del proyecto
YOLO_REPO = ROOT / "yolov5"  # carpeta donde está hubconf.py

# Ajusta expN según tu entrenamiento real
WEIGHTS = YOLO_REPO / "runs" / "train" / "exp9" / "weights" / "best.pt"
DATA_YAML = ROOT / "dataset" / "data.yaml"
FOLDER_SOURCE = ROOT / "dataset" / "test" / "images"

DEVICE = "cpu"   # o "0" si tienes GPU
IMG_SIZE = 640
CONF_THRES = 0.25

print("Repo YOLOv5:", YOLO_REPO.resolve())
print("Modelo:", WEIGHTS.resolve())
print("Data.yaml:", DATA_YAML.resolve())


Repo YOLOv5: C:\Users\OPEN SERVICE EIRL\Documents\UPN\Ciclo 7\Machine learning\Proyecto_Final_Machine Learning\Proyecto Final_Machine learning\yolov5
Modelo: C:\Users\OPEN SERVICE EIRL\Documents\UPN\Ciclo 7\Machine learning\Proyecto_Final_Machine Learning\Proyecto Final_Machine learning\yolov5\runs\train\exp9\weights\best.pt
Data.yaml: C:\Users\OPEN SERVICE EIRL\Documents\UPN\Ciclo 7\Machine learning\Proyecto_Final_Machine Learning\Proyecto Final_Machine learning\dataset\data.yaml


Cargar el modelo

In [2]:
import torch

assert (YOLO_REPO / "hubconf.py").exists(), f"No encuentro hubconf.py en {YOLO_REPO}"
assert WEIGHTS.exists(), f"No encuentro el modelo: {WEIGHTS}"

model = torch.hub.load(
    str(YOLO_REPO),   # repo local (no 'ultralytics/yolov5')
    'custom',
    path=str(WEIGHTS),
    source='local',
    verbose=False
)
model.conf = CONF_THRES
model.to(DEVICE)

model


YOLOv5  2025-9-27 Python-3.12.5 torch-2.8.0+cpu CPU

Fusing layers... 
Model summary: 157 layers, 7020913 parameters, 0 gradients, 15.8 GFLOPs
Adding AutoShape... 


AutoShape(
  (model): DetectMultiBackend(
    (model): DetectionModel(
      (model): Sequential(
        (0): Conv(
          (conv): Conv2d(3, 32, kernel_size=(6, 6), stride=(2, 2), padding=(2, 2))
          (act): SiLU(inplace=True)
        )
        (1): Conv(
          (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (act): SiLU(inplace=True)
        )
        (2): C3(
          (cv1): Conv(
            (conv): Conv2d(64, 32, kernel_size=(1, 1), stride=(1, 1))
            (act): SiLU(inplace=True)
          )
          (cv2): Conv(
            (conv): Conv2d(64, 32, kernel_size=(1, 1), stride=(1, 1))
            (act): SiLU(inplace=True)
          )
          (cv3): Conv(
            (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
            (act): SiLU(inplace=True)
          )
          (m): Sequential(
            (0): Bottleneck(
              (cv1): Conv(
                (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1))
  

Inferencia en una imagen

In [6]:
import cv2, matplotlib.pyplot as plt

# Elige una imagen del test set
img_path = next(FOLDER_SOURCE.glob("*.*"))
print("Imagen usada:", img_path)

# Inferencia
res = model(str(img_path), size=IMG_SIZE)
df = res.pandas().xyxy[0]
display(df)

# Mostrar
bgr = cv2.imread(str(img_path))
for _, r in df.iterrows():
    x1,y1,x2,y2 = map(int, [r.xmin, r.ymin, r.xmax, r.ymax])
    cv2.rectangle(bgr, (x1,y1), (x2,y2), (0,255,0), 2)
    label = f"{r['name']} {r['confidence']:.2f}"
    cv2.putText(bgr, label, (x1, max(0,y1-5)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (30,220,30), 2)

rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(8,6))
plt.imshow(rgb)
plt.axis("off")
plt.show()


Imagen usada: ..\dataset\test\images\-2-_jpg.rf.19693f31bfb5b401288f2b9e2dc51ba7.jpg


c:\Users\OPEN SERVICE EIRL\Documents\UPN\Ciclo 7\Machine learning\Proyecto_Final_Machine Learning\Proyecto Final_Machine learning\prueba\..\yolov5\models\common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):


,xmin,ymin,xmax,ymax,confidence,class,name
0,47.997726,320.144135,317.523071,632.862793,0.890730,2,orange
1,319.530426,383.221954,596.992920,638.590820,0.888877,1,mandarin
2,213.891052,99.325394,487.795227,416.666077,0.847684,1,mandarin


Conteo por clase en carpeta

In [7]:
from collections import Counter
import pandas as pd

counts = Counter()
for p in FOLDER_SOURCE.glob("*.*"):
    r = model(str(p), size=IMG_SIZE)
    for name in r.pandas().xyxy[0]['name'].tolist():
        counts[name] += 1

pd.DataFrame(sorted(counts.items()), columns=["Clase","Detecciones"])


c:\Users\OPEN SERVICE EIRL\Documents\UPN\Ciclo 7\Machine learning\Proyecto_Final_Machine Learning\Proyecto Final_Machine learning\prueba\..\yolov5\models\common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
c:\Users\OPEN SERVICE EIRL\Documents\UPN\Ciclo 7\Machine learning\Proyecto_Final_Machine Learning\Proyecto Final_Machine learning\prueba\..\yolov5\models\common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
c:\Users\OPEN SERVICE EIRL\Documents\UPN\Ciclo 7\Machine learning\Proyecto_Final_Machine Learning\Proyecto Final_Machine learning\prueba\..\yolov5\models\common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
c:\Users\OPEN SERVICE EIRL\Docu

,Clase,Detecciones
0,grapefruit,89
1,lemon,547
2,mandarin,124
3,orange,71


In [10]:
from pathlib import Path

ROOT = Path("..")
base = (ROOT / "dataset").resolve()
val_imgs   = sorted((base / "valid" / "images").glob("*.*"))
val_lbls   = sorted((base / "valid" / "labels").glob("*.txt"))
train_lbls = sorted((base / "train" / "labels").glob("*.txt"))

print("valid/images:", len(val_imgs))
print("valid/labels:", len(val_lbls))
print("train/labels:", len(train_lbls))
print("\nPrimeras 5 imágenes de valid:")
for p in val_imgs[:5]:
    print("  ", p.name, "→ label esperado:", p.with_suffix(".txt").name)




valid/images: 555
valid/labels: 562
train/labels: 2810

Primeras 5 imágenes de valid:
   -1-_png.rf.38fbb29422abd02b5110281d00b7da6a.jpg → label esperado: -1-_png.rf.38fbb29422abd02b5110281d00b7da6a.txt
   -16-_jpg.rf.ef9937a53050f0f1b3d5afdd300b4beb.jpg → label esperado: -16-_jpg.rf.ef9937a53050f0f1b3d5afdd300b4beb.txt
   -2025-02-03T152615_192_jpg.rf.2ac2323b3f1490e2f612267bdb6c314e.jpg → label esperado: -2025-02-03T152615_192_jpg.rf.2ac2323b3f1490e2f612267bdb6c314e.txt
   -2025-02-03T152715_700_jpg.rf.4be2a1f39852d1e7c1830e2981eab8e7.jpg → label esperado: -2025-02-03T152715_700_jpg.rf.4be2a1f39852d1e7c1830e2981eab8e7.txt
   -2025-02-03T152723_629_jpg.rf.c3d4c9c1352b59e761d9dee8767f6e83.jpg → label esperado: -2025-02-03T152723_629_jpg.rf.c3d4c9c1352b59e761d9dee8767f6e83.txt


In [11]:
from pathlib import Path
from shutil import copy2

ROOT = Path("..")
base = (ROOT / "dataset").resolve()
src_labels = base / "train" / "labels"
dst_labels = base / "valid" / "labels"
dst_labels.mkdir(parents=True, exist_ok=True)

missing = []
copied  = 0

for img in (base / "valid" / "images").glob("*.*"):
    lbl_name = img.with_suffix(".txt").name
    src = src_labels / lbl_name
    dst = dst_labels / lbl_name
    if src.exists():
        copy2(src, dst)
        copied += 1
    else:
        missing.append(lbl_name)

print(f"Copiados: {copied}")
if missing:
    print(f"FALTAN {len(missing)} labels (no se encontraron en train/labels):")
    for m in missing[:30]:
        print(" -", m)


Copiados: 10
FALTAN 545 labels (no se encontraron en train/labels):
 - -1-_png.rf.38fbb29422abd02b5110281d00b7da6a.txt
 - -16-_jpg.rf.ef9937a53050f0f1b3d5afdd300b4beb.txt
 - -2025-02-03T152615_192_jpg.rf.2ac2323b3f1490e2f612267bdb6c314e.txt
 - -2025-02-03T152715_700_jpg.rf.4be2a1f39852d1e7c1830e2981eab8e7.txt
 - -2025-02-03T152723_629_jpg.rf.c3d4c9c1352b59e761d9dee8767f6e83.txt
 - -21-_jpg.rf.074778b5654b275ad9b9b35b8226dbc0.txt
 - -30-_jpg.rf.0f228a509fbc4b63150bb756cc899875.txt
 - -35-_jpg.rf.dc8b8ff333eecfeabf25a75d327b5291.txt
 - -49-_jpg.rf.b33bc9115d235f3b93c91055253585ca.txt
 - -51-_jpg.rf.c34ecac40a124941fc6a03e174ca6b9a.txt
 - -54-_jpg.rf.5b5b99a6149efc0f3c8955be98e63510.txt
 - -59-_jpg.rf.880d683e4792a017a1b7df7163bd9a0e.txt
 - -61-_jpg.rf.6aca47eef6bd0cfe038b314e737dad0b.txt
 - -64-_jpg.rf.ed3bd5dd29f46c2b9c754312c77dbd2a.txt
 - -68-_jpg.rf.7f24b1a7c231e768646646fd88b30010.txt
 - -72-_jpg.rf.de604b2921acc8a45c58d578a2a96980.txt
 - -75-_jpg.rf.ba2af9b10901c60b5274b014847cdcdb

In [12]:
import yaml

with open((ROOT/"dataset"/"data.yaml"), "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
nc = int(cfg["nc"])

bad = []
for lbl in (ROOT/"dataset"/"valid"/"labels").glob("*.txt"):
    with open(lbl, "r", encoding="utf-8") as f:
        for i, ln in enumerate(l.strip() for l in f if l.strip()):
            parts = ln.split()
            if len(parts) != 5:
                bad.append((lbl.name, i+1, "len!=5", ln)); continue
            try:
                cid = int(parts[0]); x,y,w,h = map(float, parts[1:])
            except Exception:
                bad.append((lbl.name, i+1, "parse", ln)); continue
            if not (0 <= cid < nc):
                bad.append((lbl.name, i+1, f"class {cid} out of range", ln))
            if not all(0.0 <= v <= 1.0 for v in (x,y,w,h)) or w<=0 or h<=0:
                bad.append((lbl.name, i+1, "range/size", ln))

print("Problemas:", len(bad))
for b in bad[:30]:
    print(b)


Problemas: 20
('44e24fda3ccacd26495d4cba92606169_jpg.rf.b375b5946e859735d550b71d4527ba07.txt', 1, 'len!=5', '0 0.2470023984375 0.15533980625 0.14148681093749998 0.27912621406250004 0.08633093593749999 0.3932038828125 0.0671462828125 0.4902912625 0.0695443640625 0.5315533984374999 0.081534771875 0.567961165625 0.1558753 0.6383495140625 0.237410071875 0.67961165 0.26858513125 0.686893203125 0.26618705 0.6432038828125 0.27098321406250003 0.6237864078124999 0.2805755390625 0.6165048546875 0.366906475 0.6116504859375 0.6378896875 0.567961165625 0.657074340625 0.5582524265625 0.6882494 0.4902912625 0.7026378890625 0.3762135921875 0.6882494 0.2985436890625 0.6690647484375 0.25 0.6187050359375 0.172330096875 0.5611510796875 0.1092233015625 0.5347721828124999 0.08495145625 0.48201438906250005 0.06553398125 0.4684244609375 0.06553398125 0.4100719421875 0.07038835 0.36450839375 0.08495145625 0.30935251875 0.1116504859375 0.2470023984375 0.15533980625')
('44e24fda3ccacd26495d4cba92606169_jpg.rf.b3

Validación del modelo en valid/ (mAP, P, R)

In [18]:
import subprocess
import sys
import shlex
from pathlib import Path

# Asegúrate de que las rutas sean absolutas
YOLO_REPO = Path("..") / "yolov5"
WEIGHTS   = YOLO_REPO / "runs" / "train" / "exp7" / "weights" / "best.pt"
DATA_YAML = Path("..") / "dataset" / "data.yaml"

# Verifica que las rutas existan
assert WEIGHTS.exists(), f"No se encontró el modelo: {WEIGHTS}"
assert DATA_YAML.exists(), f"No se encontró el archivo de datos: {DATA_YAML}"

# Ejecuta el comando para la validación
cmd = [
    sys.executable, str(YOLO_REPO / "val.py"),  # Ruta al archivo val.py dentro de yolov5
    "--weights", str(WEIGHTS.resolve()),  # Resolución de la ruta para asegurar que esté correcta
    "--data", str(DATA_YAML.resolve()),  # Resolución de la ruta para el archivo data.yaml
    "--img", "640",  # Tamaño de la imagen
    "--conf", "0.25",  # Umbral de confianza
    "--iou", "0.6",  # Umbral de IOU
    "--task", "val",  # Validación
    "--device", "cpu"  # Usar CPU para la validación
]

print("Ejecutando:\n", " ".join(shlex.quote(c) for c in cmd))  # Imprimir el comando para ver la ejecución

# Ejecuta el comando
proc = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8")

# Muestra la salida del proceso
print("\n--- STDOUT ---\n", proc.stdout)
print("\n--- STDERR ---\n", proc.stderr)

# Verificar si hubo algún error en la ejecución
if proc.returncode != 0:
    raise RuntimeError(f"val.py falló (ver STDERR)")
else:
    print("✅ Validación completada. Revisa: yolov5/runs/val/exp*/")




Ejecutando:
 'c:\Users\OPEN SERVICE EIRL\Documents\UPN\Ciclo 7\Machine learning\Proyecto_Final_Machine Learning\Proyecto Final_Machine learning\.venv\Scripts\python.exe' '..\yolov5\val.py' --weights 'C:\Users\OPEN SERVICE EIRL\Documents\UPN\Ciclo 7\Machine learning\Proyecto_Final_Machine Learning\Proyecto Final_Machine learning\yolov5\runs\train\exp7\weights\best.pt' --data 'C:\Users\OPEN SERVICE EIRL\Documents\UPN\Ciclo 7\Machine learning\Proyecto_Final_Machine Learning\Proyecto Final_Machine learning\dataset\data.yaml' --img 640 --conf 0.25 --iou 0.6 --task val --device cpu

--- STDOUT ---
 

--- STDERR ---
 val: data=C:\Users\OPEN SERVICE EIRL\Documents\UPN\Ciclo 7\Machine learning\Proyecto_Final_Machine Learning\Proyecto Final_Machine learning\dataset\data.yaml, weights=['C:\\Users\\OPEN SERVICE EIRL\\Documents\\UPN\\Ciclo 7\\Machine learning\\Proyecto_Final_Machine Learning\\Proyecto Final_Machine learning\\yolov5\\runs\\train\\exp7\\weights\\best.pt'], batch_size=32, imgsz=640, c

Auditor de dataset/labels 

In [17]:
import yaml, glob

with open(DATA_YAML, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

base = Path(cfg.get("path", ROOT/"dataset"))
print("Base:", base)

for split in ["train","valid","test"]:
    imgs = list((base/split/"images").glob("*.*"))
    print(f"{split}: {len(imgs)} imágenes")


Base: C:\Users\OPEN SERVICE EIRL\Documents\UPN\Ciclo 7\Machine learning\Proyecto_Final_Machine Learning\Proyecto Final_Machine learning\dataset
train: 2810 imágenes
valid: 555 imágenes
test: 235 imágenes
